# Chapter 14: What Gets Measured Gets Managed
## FinOps, Observability, and the Dollar-per-Decision Metric

**Core Claim:** As agentic systems move from experimentation to enterprise scale, the relevant unit of measurement shifts from cost-per-inference to dollar-per-decision. Without this metric, organizations optimize for the wrong variable and produce agents that are technically efficient but economically indefensible.

**What this notebook demonstrates:**
1. The \$147,000 failure — a legal tech agent with no span-level cost monitoring
2. Why cost-per-inference hides the real problem
3. Building the DpD observability pipeline incrementally (span instrumentation → DpD metric → budget envelope → circuit breaker → model routing)
4. Two Human Decision Nodes where AI proposals are rejected on architectural grounds
5. A triggerable failure exercise the reader can reproduce

---

**No API keys required.** All LLM costs are simulated with realistic token counts and pricing to demonstrate the architectural concepts.

## Cell 1 — Setup & Cost Model Configuration

We define three model tiers with realistic pricing, and a simulated document corpus.
The legal tech agent processes contracts: classify each clause, flag anomalies, produce a summary.

**Key design decision:** Pricing is defined per-token for input and output separately, reflecting the real asymmetry where output tokens cost 3–5× more than input tokens.

In [1]:
import random
import uuid
import time
import json
import copy
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# MODEL TIER PRICING (per million tokens)
# These reflect realistic 2024-2025 pricing structures.
# Output tokens cost ~3-5x more than input tokens.
# ============================================================

MODEL_PRICING = {
    "frontier": {
        "name": "claude-opus-4 (Frontier)",
        "input_per_million": 15.00,    # $15.00 per 1M input tokens
        "output_per_million": 75.00,   # $75.00 per 1M output tokens
        "cache_discount": 0.90,         # 90% discount on cached input tokens
    },
    "mid_tier": {
        "name": "claude-sonnet-4 (Mid-Tier)",
        "input_per_million": 3.00,
        "output_per_million": 15.00,
        "cache_discount": 0.90,
    },
    "lightweight": {
        "name": "claude-haiku-4 (Lightweight)",
        "input_per_million": 0.25,
        "output_per_million": 1.25,
        "cache_discount": 0.90,
    },
}

def compute_call_cost(model_tier: str, input_tokens: int, output_tokens: int, cached_tokens: int = 0) -> float:
    """Compute the cost of a single LLM call using the formula:
    C_call = n_in * P_in + n_out * P_out - n_cached * P_cache_discount
    """
    pricing = MODEL_PRICING[model_tier]
    input_cost = (input_tokens / 1_000_000) * pricing["input_per_million"]
    output_cost = (output_tokens / 1_000_000) * pricing["output_per_million"]
    cache_savings = (cached_tokens / 1_000_000) * pricing["input_per_million"] * pricing["cache_discount"]
    return input_cost + output_cost - cache_savings

# ============================================================
# DOCUMENT CORPUS
# 50 contracts. Document #38 is malformed (triggers retry loops).
# This mirrors the chapter's scenario: document 218 in a 1,000-doc
# corpus, scaled down for demonstration.
# ============================================================

NUM_DOCUMENTS = 50
MALFORMED_DOC_INDEX = 38  # 0-indexed; this is "document 218" from the chapter

@dataclass
class Document:
    doc_id: int
    title: str
    clause_count: int
    is_malformed: bool = False
    complexity: str = "standard"  # standard, complex, simple

random.seed(42)
corpus = []
for i in range(NUM_DOCUMENTS):
    complexity = random.choice(["simple"] * 5 + ["standard"] * 3 + ["complex"] * 2)
    doc = Document(
        doc_id=i,
        title=f"Contract-{i+1:04d}",
        clause_count=random.randint(5, 25),
        is_malformed=(i == MALFORMED_DOC_INDEX),
        complexity=complexity,
    )
    corpus.append(doc)

print(f"Corpus: {NUM_DOCUMENTS} documents")
print(f"Malformed document: {corpus[MALFORMED_DOC_INDEX].title} (index {MALFORMED_DOC_INDEX})")
print(f"\nComplexity distribution:")
for c in ["simple", "standard", "complex"]:
    count = sum(1 for d in corpus if d.complexity == c)
    print(f"  {c}: {count} documents")
print(f"\nModel pricing loaded:")
for tier, p in MODEL_PRICING.items():
    print(f"  {p['name']}: ${p['input_per_million']:.2f} / ${p['output_per_million']:.2f} per 1M tokens (in/out)")

Corpus: 50 documents
Malformed document: Contract-0039 (index 38)

Complexity distribution:
  simple: 28 documents
  standard: 13 documents
  complex: 9 documents

Model pricing loaded:
  claude-opus-4 (Frontier): $15.00 / $75.00 per 1M tokens (in/out)
  claude-sonnet-4 (Mid-Tier): $3.00 / $15.00 per 1M tokens (in/out)
  claude-haiku-4 (Lightweight): $0.25 / $1.25 per 1M tokens (in/out)


## Cell 2 — Span Schema & Trace Infrastructure

Every LLM call in an agentic system should emit a structured trace event. This is the observability foundation.

From the chapter:
> *"Each LLM call emits a structured trace event containing at minimum: span ID, parent span ID, model name, input token count, output token count, cached token count, computed cost, and wall-clock latency."*

The `decision_bearing` flag is the hinge on which the DpD metric turns. Not every LLM call is a "decision" — reformatting JSON or validating a schema doesn't change the agent's behavior.

In [2]:
@dataclass
class Span:
    """A single instrumented operation in the agent's execution graph.

    This schema mirrors the chapter's span definition:
    {
        "span_id": "a3f2c1",
        "parent_span_id": "root_00",
        "operation": "classify_clause_type",
        "model": "claude-sonnet-4",
        "input_tokens": 890,
        "output_tokens": 45,
        "cached_tokens": 712,
        "cost_usd": 0.00191,
        "latency_ms": 843,
        "decision_bearing": true
    }
    """
    span_id: str
    parent_span_id: str
    operation: str
    model_tier: str
    input_tokens: int
    output_tokens: int
    cached_tokens: int
    cost_usd: float
    latency_ms: int
    decision_bearing: bool
    doc_id: int
    is_retry: bool = False
    retry_number: int = 0


class TraceLog:
    """Collects all spans from an agent run. This is the observability backbone."""

    def __init__(self):
        self.spans: List[Span] = []

    def record(self, span: Span):
        self.spans.append(span)

    def total_cost(self) -> float:
        return sum(s.cost_usd for s in self.spans)

    def total_calls(self) -> int:
        return len(self.spans)

    def cost_per_inference(self) -> float:
        """The WRONG metric — average cost per LLM call."""
        if not self.spans:
            return 0.0
        return self.total_cost() / self.total_calls()

    def decision_bearing_spans(self) -> List[Span]:
        return [s for s in self.spans if s.decision_bearing]

    def dollar_per_decision(self) -> float:
        """The RIGHT metric — total cost / count of decision-bearing spans.

        DpD = (sum of ALL span costs) / (count of decision-bearing spans)

        The numerator includes ALL costs because supporting calls are real
        expenditures in service of decisions; the denominator counts only
        the decisions those costs produce.
        """
        decision_count = len(self.decision_bearing_spans())
        if decision_count == 0:
            return float('inf')
        return self.total_cost() / decision_count

    def spans_for_doc(self, doc_id: int) -> List[Span]:
        return [s for s in self.spans if s.doc_id == doc_id]

    def cost_for_doc(self, doc_id: int) -> float:
        return sum(s.cost_usd for s in self.spans_for_doc(doc_id))


print("Span schema and TraceLog defined.")
print("\nKey methods:")
print("  trace.cost_per_inference()   — the WRONG metric (avg cost per call)")
print("  trace.dollar_per_decision()  — the RIGHT metric (total cost / decision count)")

Span schema and TraceLog defined.

Key methods:
  trace.cost_per_inference()   — the WRONG metric (avg cost per call)
  trace.dollar_per_decision()  — the RIGHT metric (total cost / decision count)


## Cell 3 — The Agent Simulation Engine

This simulates the legal tech agent's execution graph for each document:

```
root_task
├── plan_subtasks        [LLM, ~1200 tokens in, ~340 out]   — decision-bearing
├── retrieve_sections    [vector_db, ~$0.002]                — non-LLM
├── classify_clause      [LLM, ~890 tokens in, ~45 out]     — DECISION-BEARING
│   ├── clarify_term     [LLM, ~1100 tokens in, ~80 out]    — support
│   └── lookup_precedent [retrieval, ~$0.001]                — non-LLM
├── format_output        [LLM, ~400 tokens in, ~200 out]    — support (NOT decision-bearing)
├── validate_schema      [LLM, ~300 tokens in, ~50 out]     — support (NOT decision-bearing)
├── flag_anomaly         [LLM, ~950 tokens in, ~120 out]    — decision-bearing
└── synthesize_summary   [LLM, ~3400 tokens in, ~560 out]   — decision-bearing
```

**Critical distinction:** Only 4 of 7 LLM calls are decision-bearing. The other 3 (clarify_term, format_output, validate_schema) are support calls that could be replaced by cheaper models or deterministic functions.

In [3]:
# ============================================================
# AGENT TASK DEFINITIONS
# Each task has a baseline token profile and decision-bearing flag.
# The flag is set at DESIGN TIME by the architect — not at runtime
# by the LLM. (See Human Decision Node #2 for why.)
# ============================================================

@dataclass
class TaskProfile:
    """Defines the expected token footprint and properties of an agent task."""
    operation: str
    base_input_tokens: int
    base_output_tokens: int
    token_variance: float          # random multiplier range (e.g., 0.15 = ±15%)
    decision_bearing: bool
    recommended_tier: str          # what tier SHOULD this run on?
    cacheable_tokens: int = 0      # tokens from system prompt / stable prefix


# The execution graph for one document
TASK_GRAPH = [
    TaskProfile("plan_subtasks",       1200, 340,  0.10, True,  "mid_tier",   500),
    TaskProfile("classify_clause",      890,  45,  0.15, True,  "frontier",   500),
    TaskProfile("clarify_term",        1100,  80,  0.20, False, "lightweight", 500),
    TaskProfile("format_output",        400, 200,  0.10, False, "lightweight",   0),
    TaskProfile("validate_schema",      300,  50,  0.05, False, "lightweight",   0),
    TaskProfile("flag_anomaly",         950, 120,  0.15, True,  "mid_tier",   500),
    TaskProfile("synthesize_summary",  3400, 560,  0.10, True,  "frontier",   500),
]

print("Agent Task Graph for Document Processing:")
print("=" * 75)
print(f"{'Operation':<22} {'Input':>6} {'Output':>7} {'Decision?':>10} {'Recommended Tier':>18}")
print("-" * 75)
for t in TASK_GRAPH:
    flag = "YES" if t.decision_bearing else "no"
    print(f"{t.operation:<22} {t.base_input_tokens:>6} {t.base_output_tokens:>7} {flag:>10} {t.recommended_tier:>18}")

decision_count = sum(1 for t in TASK_GRAPH if t.decision_bearing)
support_count = sum(1 for t in TASK_GRAPH if not t.decision_bearing)
print(f"\nDecision-bearing calls: {decision_count} | Support calls: {support_count}")
print(f"A system counting all {len(TASK_GRAPH)} calls equally HIDES the real cost per decision.")

Agent Task Graph for Document Processing:
Operation               Input  Output  Decision?   Recommended Tier
---------------------------------------------------------------------------
plan_subtasks            1200     340        YES           mid_tier
classify_clause           890      45        YES           frontier
clarify_term             1100      80         no        lightweight
format_output             400     200         no        lightweight
validate_schema           300      50         no        lightweight
flag_anomaly              950     120        YES           mid_tier
synthesize_summary       3400     560        YES           frontier

Decision-bearing calls: 4 | Support calls: 3
A system counting all 7 calls equally HIDES the real cost per decision.


In [4]:
# ============================================================
# SIMULATION ENGINE
# Processes documents through the agent's execution graph.
# Configurable: model routing, budget limits, circuit breakers.
# ============================================================

@dataclass
class AgentConfig:
    """Configuration for the agent simulation.
    Toggle these to observe different failure modes.
    """
    # Model routing
    use_tiered_routing: bool = False      # If False, ALL calls go to default_model
    default_model: str = "frontier"       # Model used when routing is disabled

    # Budget controls
    budget_per_task: Optional[float] = None   # Per-document budget ceiling ($)

    # Circuit breaker
    circuit_breaker_enabled: bool = False
    circuit_breaker_threshold: float = 2.0    # Cost-velocity ratio threshold
    circuit_breaker_window: int = 10           # Rolling window size

    # Retry behavior (for malformed documents)
    max_retries: int = 50                      # High default to show the failure
    retry_appends_history: bool = True          # Context grows with each retry

    # Caching
    enable_caching: bool = False


class BudgetExceededError(Exception):
    pass

class CircuitBreakerTripped(Exception):
    pass


def simulate_agent_run(corpus: List[Document], config: AgentConfig) -> TraceLog:
    """Run the legal tech agent across the document corpus.

    For each document:
    - Execute the task graph (plan → classify → clarify → format → validate → flag → summarize)
    - If the document is malformed, classification fails and triggers retries
    - Each retry appends full conversation history (context bloat)
    - Budget envelope and circuit breaker can interrupt if enabled
    """
    trace = TraceLog()
    random.seed(42)

    # Track baseline costs for circuit breaker
    recent_costs = []  # Rolling window of per-call costs
    baseline_costs = {}  # Mean cost per operation type from first 10 docs
    calibration_phase = True

    docs_processed = 0
    docs_skipped = 0
    budget_violations = 0
    circuit_breaker_trips = 0

    for doc in corpus:
        task_cost_so_far = 0.0
        doc_spans = []
        root_id = str(uuid.uuid4())[:8]
        doc_aborted = False
        accumulated_history_tokens = 0  # Simulates context bloat on retries

        for task in TASK_GRAPH:
            if doc_aborted:
                break

            # Determine which model to use
            if config.use_tiered_routing:
                model_tier = task.recommended_tier
            else:
                model_tier = config.default_model

            # Handle malformed document retries on classify_clause
            if doc.is_malformed and task.operation == "classify_clause":
                for retry_num in range(config.max_retries):
                    # Context bloat: each retry appends prior history
                    if config.retry_appends_history:
                        accumulated_history_tokens += task.base_input_tokens + task.base_output_tokens

                    variance = 1.0 + random.uniform(-task.token_variance, task.token_variance)
                    input_tokens = int((task.base_input_tokens + accumulated_history_tokens) * variance)
                    output_tokens = int(task.base_output_tokens * variance * 1.5)  # Error traces are verbose
                    cached = int(task.cacheable_tokens * config.enable_caching * 0.9)

                    cost = compute_call_cost(model_tier, input_tokens, output_tokens, cached)

                    # --- BUDGET ENVELOPE CHECK ---
                    if config.budget_per_task is not None:
                        if task_cost_so_far + cost > config.budget_per_task:
                            budget_violations += 1
                            doc_aborted = True
                            break

                    # --- CIRCUIT BREAKER CHECK ---
                    if config.circuit_breaker_enabled and not calibration_phase:
                        op_baseline = baseline_costs.get(task.operation, cost)
                        if op_baseline > 0:
                            velocity_ratio = cost / op_baseline
                            if velocity_ratio > config.circuit_breaker_threshold:
                                circuit_breaker_trips += 1
                                doc_aborted = True
                                break

                    span = Span(
                        span_id=str(uuid.uuid4())[:8],
                        parent_span_id=root_id,
                        operation=task.operation,
                        model_tier=model_tier,
                        input_tokens=input_tokens,
                        output_tokens=output_tokens,
                        cached_tokens=cached,
                        cost_usd=cost,
                        latency_ms=int(input_tokens * 0.8 + output_tokens * 2.5),
                        decision_bearing=task.decision_bearing,
                        doc_id=doc.doc_id,
                        is_retry=True,
                        retry_number=retry_num + 1,
                    )
                    trace.record(span)
                    task_cost_so_far += cost
                    recent_costs.append(cost)

                # Malformed doc never succeeds — skip remaining tasks
                doc_aborted = True
                continue

            # Normal (non-retry) execution
            variance = 1.0 + random.uniform(-task.token_variance, task.token_variance)
            input_tokens = int(task.base_input_tokens * variance)
            output_tokens = int(task.base_output_tokens * variance)
            cached = int(task.cacheable_tokens * config.enable_caching * 0.9)

            cost = compute_call_cost(model_tier, input_tokens, output_tokens, cached)

            # --- BUDGET ENVELOPE CHECK ---
            if config.budget_per_task is not None:
                if task_cost_so_far + cost > config.budget_per_task:
                    budget_violations += 1
                    doc_aborted = True
                    break

            span = Span(
                span_id=str(uuid.uuid4())[:8],
                parent_span_id=root_id,
                operation=task.operation,
                model_tier=model_tier,
                input_tokens=input_tokens,
                output_tokens=output_tokens,
                cached_tokens=cached,
                cost_usd=cost,
                latency_ms=int(input_tokens * 0.8 + output_tokens * 2.5),
                decision_bearing=task.decision_bearing,
                doc_id=doc.doc_id,
                is_retry=False,
                retry_number=0,
            )
            trace.record(span)
            task_cost_so_far += cost
            recent_costs.append(cost)

            # Build baseline during calibration (first 10 docs)
            if doc.doc_id < 10:
                if task.operation not in baseline_costs:
                    baseline_costs[task.operation] = []
                if isinstance(baseline_costs[task.operation], list):
                    baseline_costs[task.operation].append(cost)

        if not doc_aborted:
            docs_processed += 1
        else:
            docs_skipped += 1

        # End calibration after 10 docs; compute baseline means
        if doc.doc_id == 9:
            calibration_phase = False
            for op, costs in baseline_costs.items():
                if isinstance(costs, list):
                    baseline_costs[op] = sum(costs) / len(costs)

    return trace, docs_processed, docs_skipped, budget_violations, circuit_breaker_trips


print("Agent simulation engine ready.")
print("Configure AgentConfig to toggle: model routing, budget limits, circuit breakers.")

Agent simulation engine ready.
Configure AgentConfig to toggle: model routing, budget limits, circuit breakers.


## Cell 4 — Scenario 1: No Guardrails (The \$147K Failure)

**What happens:** The agent runs with a frontier model for every call, no budget ceiling, no circuit breaker. When it hits the malformed document, retry logic kicks in with growing context windows. Cost-per-inference looks "normal" because it's averaged across hundreds of calls. But the malformed document's retries silently consume a massive share of the budget.

**The architectural failure:** No span-level cost instrumentation → no signal to differentiate normal calls from 48× cost anomalies → runaway spend.

In [5]:
# Scenario 1: No guardrails — frontier model everywhere, no budget, no circuit breaker
config_no_guardrails = AgentConfig(
    use_tiered_routing=False,
    default_model="frontier",
    budget_per_task=None,
    circuit_breaker_enabled=False,
    max_retries=50,  # Retry up to 50 times (simulating overnight runaway)
    retry_appends_history=True,
    enable_caching=False,
)

trace_1, processed_1, skipped_1, budget_v_1, cb_trips_1 = simulate_agent_run(corpus, config_no_guardrails)

print("=" * 65)
print("SCENARIO 1: NO GUARDRAILS (The $147K Failure Mode)")
print("=" * 65)
print(f"Documents processed successfully: {processed_1}")
print(f"Documents aborted:               {skipped_1}")
print(f"Total LLM calls:                 {trace_1.total_calls()}")
print(f"Total cost:                      ${trace_1.total_cost():.2f}")
print(f"")
print(f"--- THE TWO METRICS ---")
print(f"Cost-per-inference (WRONG):       ${trace_1.cost_per_inference():.4f}")
print(f"Dollar-per-decision (RIGHT):      ${trace_1.dollar_per_decision():.4f}")
print(f"")
print(f"The gap between these two numbers is where the money disappears.")
print(f"")

# Show the malformed document's cost specifically
malformed_spans = trace_1.spans_for_doc(MALFORMED_DOC_INDEX)
malformed_cost = trace_1.cost_for_doc(MALFORMED_DOC_INDEX)
normal_doc_costs = [trace_1.cost_for_doc(i) for i in range(NUM_DOCUMENTS) if i != MALFORMED_DOC_INDEX and trace_1.cost_for_doc(i) > 0]
avg_normal_cost = sum(normal_doc_costs) / len(normal_doc_costs) if normal_doc_costs else 0

print(f"--- MALFORMED DOCUMENT ANALYSIS ---")
print(f"Document {corpus[MALFORMED_DOC_INDEX].title}:")
print(f"  Retry attempts:    {len(malformed_spans)}")
print(f"  Cost for this doc: ${malformed_cost:.2f}")
print(f"  Avg normal doc:    ${avg_normal_cost:.4f}")
print(f"  Cost multiplier:   {malformed_cost/avg_normal_cost:.0f}× a normal document")
print(f"  % of total spend:  {malformed_cost/trace_1.total_cost()*100:.1f}%")
print(f"")
print(f"One malformed document consumed {malformed_cost/trace_1.total_cost()*100:.1f}% of the entire budget.")
print(f"Cost-per-inference HIDES this because it averages the spike across all calls.")

SCENARIO 1: NO GUARDRAILS (The $147K Failure Mode)
Documents processed successfully: 49
Documents aborted:               1
Total LLM calls:                 394
Total cost:                      $30.07

--- THE TWO METRICS ---
Cost-per-inference (WRONG):       $0.0763
Dollar-per-decision (RIGHT):      $0.1218

The gap between these two numbers is where the money disappears.

--- MALFORMED DOCUMENT ANALYSIS ---
Document Contract-0039:
  Retry attempts:    51
  Cost for this doc: $18.89
  Avg normal doc:    $0.2282
  Cost multiplier:   83× a normal document
  % of total spend:  62.8%

One malformed document consumed 62.8% of the entire budget.
Cost-per-inference HIDES this because it averages the spike across all calls.


In [6]:
# Visualize the cost spike: per-call cost over time
# This is what you'd see on a span-level dashboard — the spike is obvious.
# On a cost-per-inference dashboard? It's invisible.

print("\nPER-CALL COST TIMELINE (showing the spike)")
print("=" * 65)
print("Each bar = one LLM call. Height = cost. Malformed doc retries marked with !!!")
print()

max_cost = max(s.cost_usd for s in trace_1.spans)
bar_width = 50

# Show a window around the malformed document
malformed_start = None
for i, s in enumerate(trace_1.spans):
    if s.doc_id == MALFORMED_DOC_INDEX:
        malformed_start = max(0, i - 5)
        break

if malformed_start is not None:
    window = trace_1.spans[malformed_start:malformed_start + 60]
    print(f"  Showing calls {malformed_start} to {malformed_start + len(window)}:")
    print()
    for i, s in enumerate(window):
        bar_len = int((s.cost_usd / max_cost) * bar_width)
        bar = "█" * bar_len
        marker = " !!!" if s.doc_id == MALFORMED_DOC_INDEX else ""
        retry_info = f" (retry #{s.retry_number})" if s.is_retry else ""
        print(f"  Call {malformed_start + i:>4} | ${s.cost_usd:>8.4f} |{bar}{marker}{retry_info}")

print(f"\nThe cost-velocity ratio on the worst retry:")
worst_retry = max(malformed_spans, key=lambda s: s.cost_usd)
baseline_classify = compute_call_cost("frontier", 890, 45, 0)
print(f"  Worst retry cost: ${worst_retry.cost_usd:.4f}")
print(f"  Baseline cost:    ${baseline_classify:.6f}")
print(f"  Cost-velocity ratio: {worst_retry.cost_usd / baseline_classify:.1f}×")
print(f"\n  A circuit breaker at 2.0× would have caught this after retry #2.")


PER-CALL COST TIMELINE (showing the spike)
Each bar = one LLM call. Height = cost. Malformed doc retries marked with !!!

  Showing calls 261 to 321:

  Call  261 | $  0.0259 |█
  Call  262 | $  0.0204 |█
  Call  263 | $  0.0079 |
  Call  264 | $  0.0256 |█
  Call  265 | $  0.0967 |██████
  Call  266 | $  0.0444 |██ !!!
  Call  267 | $  0.0371 |██ !!! (retry #1)
  Call  268 | $  0.0486 |███ !!! (retry #2)
  Call  269 | $  0.0515 |███ !!! (retry #3)
  Call  270 | $  0.0815 |█████ !!! (retry #4)
  Call  271 | $  0.0832 |█████ !!! (retry #5)
  Call  272 | $  0.1075 |██████ !!! (retry #6)
  Call  273 | $  0.1319 |████████ !!! (retry #7)
  Call  274 | $  0.1163 |███████ !!! (retry #8)
  Call  275 | $  0.1279 |████████ !!! (retry #9)
  Call  276 | $  0.1399 |████████ !!! (retry #10)
  Call  277 | $  0.1754 |███████████ !!! (retry #11)
  Call  278 | $  0.1739 |███████████ !!! (retry #12)
  Call  279 | $  0.2070 |█████████████ !!! (retry #13)
  Call  280 | $  0.2287 |██████████████ !!! (retry

## Cell 5 — Scenario 2: Add Budget Envelope

From the chapter:
> *"Had the agent been initialized with a per-task budget of \$1.50 — roughly double the expected \$0.08, providing margin for legitimate complexity variation — the budget envelope would have interrupted the anomalous retry sequence at the third attempt."*

We add a simple per-document budget ceiling. This prevents disaster but does NOT prevent waste — the agent still uses the frontier model for everything.

In [7]:
# Scenario 2: Add budget envelope only
config_budget_only = AgentConfig(
    use_tiered_routing=False,
    default_model="frontier",
    budget_per_task=1.50,           # Per-document budget ceiling
    circuit_breaker_enabled=False,
    max_retries=50,
    retry_appends_history=True,
    enable_caching=False,
)

trace_2, processed_2, skipped_2, budget_v_2, cb_trips_2 = simulate_agent_run(corpus, config_budget_only)

print("=" * 65)
print("SCENARIO 2: BUDGET ENVELOPE ($1.50 per document)")
print("=" * 65)
print(f"Documents processed successfully: {processed_2}")
print(f"Documents aborted (budget):      {skipped_2}")
print(f"Budget violations caught:         {budget_v_2}")
print(f"Total LLM calls:                 {trace_2.total_calls()}")
print(f"Total cost:                      ${trace_2.total_cost():.2f}")
print(f"")
print(f"Cost-per-inference:               ${trace_2.cost_per_inference():.4f}")
print(f"Dollar-per-decision:              ${trace_2.dollar_per_decision():.4f}")
print(f"")
print(f"vs. Scenario 1 (no guardrails):   ${trace_1.total_cost():.2f}")
print(f"Savings:                          ${trace_1.total_cost() - trace_2.total_cost():.2f}")
print(f"")
print(f"Budget envelopes PREVENT DISASTER but don't prevent WASTE.")
print(f"The agent still uses the frontier model for formatting and validation.")

SCENARIO 2: BUDGET ENVELOPE ($1.50 per document)
Documents processed successfully: 49
Documents aborted (budget):      1
Budget violations caught:         1
Total LLM calls:                 356
Total cost:                      $12.54

Cost-per-inference:               $0.0352
Dollar-per-decision:              $0.0600

vs. Scenario 1 (no guardrails):   $30.07
Savings:                          $17.53

Budget envelopes PREVENT DISASTER but don't prevent WASTE.
The agent still uses the frontier model for formatting and validation.


## Cell 6 — Scenario 3: Add Circuit Breaker

The cost-velocity circuit breaker monitors the ratio of actual per-call cost to baseline:

$$\text{cost\_velocity\_ratio} = \frac{\bar{C}_{\text{recent}}}{\bar{C}_{\text{baseline}}}$$

A ratio above the threshold triggers graceful degradation — stop the current task, log the anomaly, escalate to human review, and move on.

In [8]:
# Scenario 3: Budget envelope + Circuit breaker
config_with_cb = AgentConfig(
    use_tiered_routing=False,
    default_model="frontier",
    budget_per_task=1.50,
    circuit_breaker_enabled=True,
    circuit_breaker_threshold=2.0,   # Trip when cost > 2× baseline
    circuit_breaker_window=10,
    max_retries=50,
    retry_appends_history=True,
    enable_caching=False,
)

trace_3, processed_3, skipped_3, budget_v_3, cb_trips_3 = simulate_agent_run(corpus, config_with_cb)

print("=" * 65)
print("SCENARIO 3: BUDGET ENVELOPE + CIRCUIT BREAKER (2.0× threshold)")
print("=" * 65)
print(f"Documents processed successfully: {processed_3}")
print(f"Documents aborted:               {skipped_3}")
print(f"  - Budget violations:           {budget_v_3}")
print(f"  - Circuit breaker trips:       {cb_trips_3}")
print(f"Total LLM calls:                 {trace_3.total_calls()}")
print(f"Total cost:                      ${trace_3.total_cost():.2f}")
print(f"")
print(f"Cost-per-inference:               ${trace_3.cost_per_inference():.4f}")
print(f"Dollar-per-decision:              ${trace_3.dollar_per_decision():.4f}")
print(f"")
print(f"The circuit breaker catches the anomaly FASTER than the budget ceiling.")
print(f"Budget envelope is the safety net. Circuit breaker is the early warning.")

SCENARIO 3: BUDGET ENVELOPE + CIRCUIT BREAKER (2.0× threshold)
Documents processed successfully: 49
Documents aborted:               1
  - Budget violations:           0
  - Circuit breaker trips:       1
Total LLM calls:                 344
Total cost:                      $11.21

Cost-per-inference:               $0.0326
Dollar-per-decision:              $0.0569

The circuit breaker catches the anomaly FASTER than the budget ceiling.
Budget envelope is the safety net. Circuit breaker is the early warning.


---

## Cell 7 — MANDATORY HUMAN DECISION NODE #1

### The Fixed vs. Adaptive Threshold Question

This is a point where AI-generated content proposed a specific architectural choice that requires human verification before proceeding.

In [9]:
# ============================================================
# ██  MANDATORY HUMAN DECISION NODE #1  ██
# ============================================================
#
# AI PROPOSAL:
#   "Use a fixed cost-velocity circuit breaker threshold of 2.0×
#    across all task types."
#
# ARCHITECTURAL PROBLEM:
#   A fixed threshold assumes all tasks have similar cost variance.
#   They do not.
#
#   - classify_clause: baseline ~$0.0025, variance ±15%
#     → A 2.0× threshold trips at $0.005. Appropriate.
#
#   - synthesize_summary: baseline ~$0.08, variance ±40%
#     → Normal calls can cost $0.112. A 2.0× threshold trips
#       at $0.16 — false alarms on legitimately complex docs.
#
# HUMAN DECISION:
#   REJECTED the fixed 2.0× universal threshold.
#   IMPLEMENTED per-task-type adaptive thresholds calibrated
#   from the variance observed during the first 10 documents
#   (the calibration phase).
#
#   Threshold = mean + (N × std_dev), where N is configurable.
#   This catches genuine anomalies without triggering on
#   naturally high-variance tasks.
#
# WHY THIS MATTERS:
#   A fixed threshold is itself an architectural error — it
#   trades false negatives on low-variance tasks for false
#   positives on high-variance tasks. The AI proposed it
#   because a single number is simpler to explain, not because
#   it's architecturally correct.
# ============================================================

# Demonstrate the problem with a fixed threshold
print("WHY A FIXED 2.0× THRESHOLD IS ARCHITECTURALLY WRONG")
print("=" * 65)

# Simulated cost distributions per task type (from calibration)
import statistics

task_cost_samples = {
    "classify_clause": [compute_call_cost("frontier", int(890 * (1 + random.uniform(-0.15, 0.15))),
                         int(45 * (1 + random.uniform(-0.15, 0.15))), 0) for _ in range(20)],
    "synthesize_summary": [compute_call_cost("frontier", int(3400 * (1 + random.uniform(-0.30, 0.30))),
                           int(560 * (1 + random.uniform(-0.30, 0.30))), 0) for _ in range(20)],
    "format_output": [compute_call_cost("frontier", int(400 * (1 + random.uniform(-0.10, 0.10))),
                      int(200 * (1 + random.uniform(-0.10, 0.10))), 0) for _ in range(20)],
}

print(f"{'Task':<22} {'Mean':>10} {'Std Dev':>10} {'CV':>8} {'Fixed 2.0×':>12} {'Adaptive':>12}")
print("-" * 76)

for task, costs in task_cost_samples.items():
    mean = statistics.mean(costs)
    std = statistics.stdev(costs)
    cv = std / mean  # Coefficient of variation
    fixed_threshold = mean * 2.0
    adaptive_threshold = mean + (3 * std)  # 3 sigma
    print(f"{task:<22} ${mean:>8.5f} ${std:>8.5f} {cv:>7.1%} ${fixed_threshold:>10.5f} ${adaptive_threshold:>10.5f}")

print(f"\nCV = Coefficient of Variation (std/mean). Higher CV = wider natural spread.")
print(f"\nThe fixed threshold treats classify_clause and synthesize_summary the same.")
print(f"The adaptive threshold respects each task's actual variance profile.")
print(f"\n→ DECISION: Use adaptive (mean + 3σ) per task type, calibrated from first 10 docs.")

WHY A FIXED 2.0× THRESHOLD IS ARCHITECTURALLY WRONG
Task                         Mean    Std Dev       CV   Fixed 2.0×     Adaptive
----------------------------------------------------------------------------
classify_clause        $ 0.01665 $ 0.00103    6.2% $   0.03331 $   0.01975
synthesize_summary     $ 0.09086 $ 0.01220   13.4% $   0.18171 $   0.12746
format_output          $ 0.02108 $ 0.00094    4.5% $   0.04215 $   0.02389

CV = Coefficient of Variation (std/mean). Higher CV = wider natural spread.

The fixed threshold treats classify_clause and synthesize_summary the same.
The adaptive threshold respects each task's actual variance profile.

→ DECISION: Use adaptive (mean + 3σ) per task type, calibrated from first 10 docs.


---

## Cell 8 — Scenario 4: Add Tiered Model Routing

This is the highest-leverage cost optimization. From the chapter:

> *"An agentic system routing every call to the same model regardless of task complexity is economically equivalent to sending every package by overnight express, including the ones that could go standard mail."*

Decision-bearing calls → frontier model. Support calls → lightweight model.

In [10]:
# Scenario 4: Full pipeline — tiered routing + budget + circuit breaker
config_full_pipeline = AgentConfig(
    use_tiered_routing=True,          # Route by task.recommended_tier
    default_model="frontier",
    budget_per_task=1.50,
    circuit_breaker_enabled=True,
    circuit_breaker_threshold=2.0,
    circuit_breaker_window=10,
    max_retries=50,
    retry_appends_history=True,
    enable_caching=True,
)

trace_4, processed_4, skipped_4, budget_v_4, cb_trips_4 = simulate_agent_run(corpus, config_full_pipeline)

print("=" * 65)
print("SCENARIO 4: FULL PIPELINE (Routing + Budget + Circuit Breaker + Cache)")
print("=" * 65)
print(f"Documents processed successfully: {processed_4}")
print(f"Documents aborted:               {skipped_4}")
print(f"Total LLM calls:                 {trace_4.total_calls()}")
print(f"Total cost:                      ${trace_4.total_cost():.2f}")
print(f"")
print(f"Cost-per-inference:               ${trace_4.cost_per_inference():.6f}")
print(f"Dollar-per-decision:              ${trace_4.dollar_per_decision():.4f}")
print(f"")
print(f"--- MODEL USAGE BREAKDOWN ---")
tier_costs = {}
tier_counts = {}
for s in trace_4.spans:
    tier_costs[s.model_tier] = tier_costs.get(s.model_tier, 0) + s.cost_usd
    tier_counts[s.model_tier] = tier_counts.get(s.model_tier, 0) + 1

for tier in ["frontier", "mid_tier", "lightweight"]:
    if tier in tier_costs:
        pct = tier_costs[tier] / trace_4.total_cost() * 100
        print(f"  {MODEL_PRICING[tier]['name']:<35} {tier_counts[tier]:>4} calls  ${tier_costs[tier]:>8.4f}  ({pct:.1f}%)")

print(f"\nFrontier model reserved for decision-bearing calls only.")
print(f"Support calls (format, validate, clarify) handled by lightweight model.")

SCENARIO 4: FULL PIPELINE (Routing + Budget + Circuit Breaker + Cache)
Documents processed successfully: 49
Documents aborted:               1
Total LLM calls:                 344
Total cost:                      $5.35

Cost-per-inference:               $0.015546
Dollar-per-decision:              $0.0271

--- MODEL USAGE BREAKDOWN ---
  claude-opus-4 (Frontier)              98 calls  $  4.7688  (89.2%)
  claude-sonnet-4 (Mid-Tier)            99 calls  $  0.5417  (10.1%)
  claude-haiku-4 (Lightweight)         147 calls  $  0.0373  (0.7%)

Frontier model reserved for decision-bearing calls only.
Support calls (format, validate, clarify) handled by lightweight model.


---

## Cell 9 — MANDATORY HUMAN DECISION NODE #2

### How to Classify Decision-Bearing Calls: LLM Self-Classification vs. HITL Calibration

In [11]:
# ============================================================
# ██  MANDATORY HUMAN DECISION NODE #2  ██
# ============================================================
#
# AI PROPOSAL:
#   "Use the LLM itself to self-classify each call as
#    decision-bearing or not, via a classification prompt
#    appended after each call."
#
# REJECTED BECAUSE:
#   This adds an LLM call to classify every LLM call.
#   You're spending tokens to decide if the tokens you just
#   spent were worth tracking. This violates the chapter's
#   core principle: minimize non-decision-bearing LLM usage.
#
#   For a 7-call task graph, this adds 7 classification calls
#   at ~200 tokens each. Over 50 documents = 350 extra calls
#   = ~70,000 tokens spent purely on self-classification.
#
# IMPLEMENTED INSTEAD:
#   A two-phase approach:
#   1. DESIGN-TIME TAGGING for obvious cases — the architect
#      knows classify_clause is a decision and format_output
#      is not. Tag these in the task definition. Zero cost.
#
#   2. HUMAN-IN-THE-LOOP (HITL) CALIBRATION for ambiguous
#      cases — during the first N documents, flag spans where
#      decision-bearing status is uncertain. A human reviews
#      the sample, classifies them, and those labels become
#      ground truth for production. One-time human cost,
#      not a per-call LLM cost compounding forever.
#
# ============================================================


# Demonstrate the cost of LLM self-classification
SELF_CLASSIFY_INPUT_TOKENS = 200
SELF_CLASSIFY_OUTPUT_TOKENS = 20

total_calls_in_run = trace_4.total_calls()
self_classify_cost_per_call = compute_call_cost("lightweight", SELF_CLASSIFY_INPUT_TOKENS, SELF_CLASSIFY_OUTPUT_TOKENS)
total_self_classify_cost = self_classify_cost_per_call * total_calls_in_run

print("COST OF AI's PROPOSAL: LLM Self-Classification")
print("=" * 65)
print(f"Calls in run:                     {total_calls_in_run}")
print(f"Self-classify cost per call:       ${self_classify_cost_per_call:.6f}")
print(f"Total self-classification cost:    ${total_self_classify_cost:.4f}")
print(f"Pipeline total cost:               ${trace_4.total_cost():.4f}")
print(f"Self-classify as % of pipeline:    {total_self_classify_cost/trace_4.total_cost()*100:.1f}%")
print(f"")
print(f"At scale (10,000 docs/day):")
daily_calls = total_calls_in_run * (10000 / NUM_DOCUMENTS)
daily_self_classify = self_classify_cost_per_call * daily_calls
monthly_self_classify = daily_self_classify * 30
print(f"  Daily self-classification cost:   ${daily_self_classify:.2f}")
print(f"  Monthly self-classification cost: ${monthly_self_classify:.2f}")
print(f"")
print(f"IMPLEMENTED INSTEAD: Design-time tagging + HITL calibration")
print(f"  Cost: $0.00 per call (tags are set once in code)")
print(f"  Human review: ~30 minutes during initial calibration")
print(f"  Recurring cost: $0.00")

COST OF AI's PROPOSAL: LLM Self-Classification
Calls in run:                     344
Self-classify cost per call:       $0.000075
Total self-classification cost:    $0.0258
Pipeline total cost:               $5.3479
Self-classify as % of pipeline:    0.5%

At scale (10,000 docs/day):
  Daily self-classification cost:   $5.16
  Monthly self-classification cost: $154.80

IMPLEMENTED INSTEAD: Design-time tagging + HITL calibration
  Cost: $0.00 per call (tags are set once in code)
  Human review: ~30 minutes during initial calibration
  Recurring cost: $0.00


In [12]:
# ============================================================
# HITL CALIBRATION SIMULATION
# During the first 10 documents, identify ambiguous spans
# and present them for human review.
# ============================================================

# In the real system, these would be spans where the architect
# was uncertain about the decision_bearing flag.
# Here we simulate the review interface.

ambiguous_cases = [
    {
        "operation": "clarify_term",
        "description": "LLM call to resolve an ambiguous legal term before classification",
        "example_input": "'indemnification' in context of limitation-of-liability clause",
        "current_tag": "support (non-decision)",
        "argument_for_decision": "The clarification changes how the clause is classified",
        "argument_against": "It's a lookup step — the classification call makes the actual decision",
    },
    {
        "operation": "plan_subtasks",
        "description": "LLM decomposes the document into subtasks for processing",
        "example_input": "Contract with 15 clauses, 3 appendices",
        "current_tag": "decision-bearing",
        "argument_for_decision": "The plan determines which clauses get processed and in what order",
        "argument_against": "For standard contracts, the plan is always the same",
    },
    {
        "operation": "flag_anomaly",
        "description": "LLM reviews classified clauses and flags unusual patterns",
        "example_input": "Non-standard termination clause with unusual penalty structure",
        "current_tag": "decision-bearing",
        "argument_for_decision": "Flagging determines whether a human lawyer reviews the clause",
        "argument_against": "Could be rule-based for common anomaly patterns",
    },
]

print("=" * 65)
print("HUMAN-IN-THE-LOOP CALIBRATION: Decision-Bearing Classification")
print("=" * 65)
print(f"\nReviewing {len(ambiguous_cases)} ambiguous span types from calibration phase:\n")

for i, case in enumerate(ambiguous_cases, 1):
    print(f"--- Case {i}: {case['operation']} ---")
    print(f"  Description: {case['description']}")
    print(f"  Example:     {case['example_input']}")
    print(f"  Current tag: {case['current_tag']}")
    print(f"  For decision:    {case['argument_for_decision']}")
    print(f"  Against:         {case['argument_against']}")
    print()

print("HUMAN REVIEWER DECISIONS:")
print("  clarify_term   → CONFIRMED as support (the classification call makes the decision)")
print("  plan_subtasks  → RECLASSIFIED as support for standard contracts, decision for complex")
print("  flag_anomaly   → CONFIRMED as decision-bearing (triggers human lawyer review)")
print()
print("These labels are now locked in for production. No per-call LLM cost.")
print("The human decided once. The system applies it forever.")

HUMAN-IN-THE-LOOP CALIBRATION: Decision-Bearing Classification

Reviewing 3 ambiguous span types from calibration phase:

--- Case 1: clarify_term ---
  Description: LLM call to resolve an ambiguous legal term before classification
  Example:     'indemnification' in context of limitation-of-liability clause
  Current tag: support (non-decision)
  For decision:    The clarification changes how the clause is classified
  Against:         It's a lookup step — the classification call makes the actual decision

--- Case 2: plan_subtasks ---
  Description: LLM decomposes the document into subtasks for processing
  Example:     Contract with 15 clauses, 3 appendices
  Current tag: decision-bearing
  For decision:    The plan determines which clauses get processed and in what order
  Against:         For standard contracts, the plan is always the same

--- Case 3: flag_anomaly ---
  Description: LLM reviews classified clauses and flags unusual patterns
  Example:     Non-standard termination 

---

## Cell 10 — Comparison Dashboard: All Four Scenarios

Side-by-side comparison showing the incremental value of each architectural component.

In [13]:
# ============================================================
# COMPARISON DASHBOARD: ALL FOUR SCENARIOS
# ============================================================

scenarios = [
    ("1. No Guardrails",      trace_1, processed_1, skipped_1),
    ("2. Budget Envelope",    trace_2, processed_2, skipped_2),
    ("3. + Circuit Breaker",  trace_3, processed_3, skipped_3),
    ("4. + Tiered Routing",   trace_4, processed_4, skipped_4),
]

print("=" * 80)
print("SCENARIO COMPARISON DASHBOARD")
print("=" * 80)
print(f"{'Scenario':<25} {'Total Cost':>12} {'CPI':>10} {'DpD':>10} {'Calls':>8} {'Processed':>10}")
print("-" * 80)

for name, trace, proc, skip in scenarios:
    print(f"{name:<25} ${trace.total_cost():>10.2f} ${trace.cost_per_inference():>8.4f} ${trace.dollar_per_decision():>8.4f} {trace.total_calls():>8} {proc:>7}/{NUM_DOCUMENTS}")

print("-" * 80)
print(f"\nCPI = Cost-Per-Inference (the wrong metric)")
print(f"DpD = Dollar-Per-Decision (the right metric)")

# The key insight
print(f"\n{'KEY INSIGHT':=^80}")
print(f"")
print(f"Scenario 1 CPI:  ${trace_1.cost_per_inference():.4f}  — looks reasonable!")
print(f"Scenario 1 DpD:  ${trace_1.dollar_per_decision():.4f}  — the real story.")
print(f"")
print(f"Scenario 4 CPI:  ${trace_4.cost_per_inference():.6f}  — looks tiny.")
print(f"Scenario 4 DpD:  ${trace_4.dollar_per_decision():.4f}  — genuinely controlled.")
print(f"")
print(f"CPI tells you 'each call is cheap.' DpD tells you 'each decision is worth the cost.'")
print(f"They answer different questions. Only one is the right question.")

SCENARIO COMPARISON DASHBOARD
Scenario                    Total Cost        CPI        DpD    Calls  Processed
--------------------------------------------------------------------------------
1. No Guardrails          $     30.07 $  0.0763 $  0.1218      394      49/50
2. Budget Envelope        $     12.54 $  0.0352 $  0.0600      356      49/50
3. + Circuit Breaker      $     11.21 $  0.0326 $  0.0569      344      49/50
4. + Tiered Routing       $      5.35 $  0.0155 $  0.0271      344      49/50
--------------------------------------------------------------------------------

CPI = Cost-Per-Inference (the wrong metric)
DpD = Dollar-Per-Decision (the right metric)

==================================KEY INSIGHT===================================

Scenario 1 CPI:  $0.0763  — looks reasonable!
Scenario 1 DpD:  $0.1218  — the real story.

Scenario 4 CPI:  $0.015546  — looks tiny.
Scenario 4 DpD:  $0.0271  — genuinely controlled.

CPI tells you 'each call is cheap.' DpD tells you 'each de

In [14]:
# ============================================================
# VISUAL: Cost breakdown by document
# Shows how the malformed document dominates Scenario 1
# ============================================================

print("\nPER-DOCUMENT COST DISTRIBUTION")
print("=" * 65)

for name, trace, _, _ in scenarios:
    doc_costs = []
    for i in range(NUM_DOCUMENTS):
        dc = trace.cost_for_doc(i)
        if dc > 0:
            doc_costs.append((i, dc))

    if not doc_costs:
        continue

    max_doc_cost = max(c for _, c in doc_costs)

    print(f"\n{name}:")
    # Show top 5 most expensive documents
    sorted_costs = sorted(doc_costs, key=lambda x: x[1], reverse=True)[:5]
    for doc_id, cost in sorted_costs:
        bar_len = int((cost / max_doc_cost) * 40)
        bar = "█" * bar_len
        malformed_marker = " ← MALFORMED" if doc_id == MALFORMED_DOC_INDEX else ""
        print(f"  Doc {doc_id:>3}: ${cost:>8.4f} |{bar}{malformed_marker}")


PER-DOCUMENT COST DISTRIBUTION

1. No Guardrails:
  Doc  38: $ 18.8916 |████████████████████████████████████████ ← MALFORMED
  Doc  37: $  0.2423 |
  Doc  39: $  0.2423 |
  Doc  28: $  0.2419 |
  Doc  10: $  0.2373 |

2. Budget Envelope:
  Doc  38: $  1.3191 |████████████████████████████████████████ ← MALFORMED
  Doc  37: $  0.2423 |███████
  Doc  44: $  0.2422 |███████
  Doc  28: $  0.2419 |███████
  Doc  49: $  0.2414 |███████

3. + Circuit Breaker:
  Doc  37: $  0.2423 |████████████████████████████████████████
  Doc  46: $  0.2423 |███████████████████████████████████████
  Doc  28: $  0.2419 |███████████████████████████████████████
  Doc  10: $  0.2373 |███████████████████████████████████████
  Doc   4: $  0.2363 |███████████████████████████████████████

4. + Tiered Routing:
  Doc  10: $  0.1184 |████████████████████████████████████████
  Doc  29: $  0.1173 |███████████████████████████████████████
  Doc  18: $  0.1170 |███████████████████████████████████████
  Doc  37: $  0.1161 |█

---

## Cell 11 — EXERCISE: Trigger the Failure Yourself

### Remove tiered model routing and observe what happens

**Instructions:**
1. In the config below, `use_tiered_routing` is set to `True` (the correct architecture)
2. Change it to `False` and re-run the cell
3. Observe: cost-per-inference stays roughly flat — but dollar-per-decision jumps dramatically
4. The circuit breaker does NOT catch this, because no individual call is anomalous

**The lesson:** DpD catches a failure mode that cost-per-inference AND the circuit breaker completely miss. The wrong metric creates a blind spot, and the blind spot is where the money disappears.

In [15]:
# ============================================================
# ▶▶▶ EXERCISE: TRIGGER THE FAILURE ◀◀◀
#
# STEP 1: Run this cell as-is (use_tiered_routing=True)
# STEP 2: Change use_tiered_routing to False
# STEP 3: Re-run and compare the DpD values
#
# QUESTION: Why doesn't the circuit breaker catch this?
# ============================================================

config_exercise = AgentConfig(
    use_tiered_routing=True,           # ← CHANGE THIS TO False TO TRIGGER FAILURE
    default_model="frontier",
    budget_per_task=1.50,
    circuit_breaker_enabled=True,
    circuit_breaker_threshold=2.0,
    circuit_breaker_window=10,
    max_retries=50,
    retry_appends_history=True,
    enable_caching=True,
)

trace_ex, proc_ex, skip_ex, bv_ex, cb_ex = simulate_agent_run(corpus, config_exercise)

print("=" * 65)
print(f"EXERCISE RESULTS (tiered_routing = {config_exercise.use_tiered_routing})")
print("=" * 65)
print(f"Total cost:            ${trace_ex.total_cost():.4f}")
print(f"Total calls:           {trace_ex.total_calls()}")
print(f"Cost-per-inference:    ${trace_ex.cost_per_inference():.6f}")
print(f"Dollar-per-decision:   ${trace_ex.dollar_per_decision():.4f}")
print(f"Circuit breaker trips: {cb_ex}")
print()

# Show per-tier breakdown
tier_costs = {}
tier_counts = {}
for s in trace_ex.spans:
    tier_costs[s.model_tier] = tier_costs.get(s.model_tier, 0) + s.cost_usd
    tier_counts[s.model_tier] = tier_counts.get(s.model_tier, 0) + 1

print("Model tier breakdown:")
for tier in ["frontier", "mid_tier", "lightweight"]:
    if tier in tier_costs:
        print(f"  {tier:<15} {tier_counts[tier]:>4} calls  ${tier_costs[tier]:.4f}")

print()
if not config_exercise.use_tiered_routing:
    print("⚠️  ALL calls routed to frontier model.")
    print("⚠️  Cost-per-inference looks similar to before — each call is individually 'normal'.")
    print("⚠️  Dollar-per-decision reveals the waste: you're paying frontier prices")
    print("    for formatting and validation that a lightweight model handles equally well.")
    print("⚠️  The circuit breaker NEVER trips because no single call is anomalous.")
    print("    This is the blind spot. DpD is the only metric that catches it.")
else:
    print("✓  Tiered routing active. Support calls use lightweight model.")
    print("   Now change use_tiered_routing to False and re-run to see the failure.")

EXERCISE RESULTS (tiered_routing = True)
Total cost:            $5.3479
Total calls:           344
Cost-per-inference:    $0.015546
Dollar-per-decision:   $0.0271
Circuit breaker trips: 1

Model tier breakdown:
  frontier          98 calls  $4.7688
  mid_tier          99 calls  $0.5417
  lightweight      147 calls  $0.0373

✓  Tiered routing active. Support calls use lightweight model.
   Now change use_tiered_routing to False and re-run to see the failure.


---

## Cell 12 — Open Question

The chapter's DpD framework optimizes for cost while holding quality constant via the constraint:

$$\text{route}(t) = \arg\min_{m \in \mathcal{M}} C_m \quad \text{subject to} \quad Q_m(t) \geq Q_{\min}(t)$$

But **who sets $Q_{\min}(t)$?**

- The engineer optimizing the cost metric sets it as low as possible to maximize savings.
- The domain expert (lawyer, doctor, analyst) who understands what a wrong decision costs would set it higher.
- These two incentives are in direct tension.

A DpD metric without a quality floor set by the right person is an optimization target that will reliably be met by degrading quality. **The quality floor is not a technical parameter — it is a governance decision.**

This is the question Chapter 14 surfaces but does not fully resolve. It sits at the intersection of engineering (what CAN we measure), economics (what SHOULD we optimize), and governance (who DECIDES the constraints).

In [16]:
# ============================================================
# SUMMARY: The Architectural Argument
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║  CHAPTER 14: WHAT GETS MEASURED GETS MANAGED               ║
║  The Architectural Argument                                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  MASTER CLAIM:                                             ║
║  Architecture is the leverage point, not the model.        ║
║                                                            ║
║  THIS CHAPTER'S INSTANCE:                                  ║
║  The unit of cost measurement determines whether an        ║
║  agentic system is economically viable at scale.           ║
║  Cost-per-inference hides compounding. Dollar-per-decision ║
║  reveals it. The metric IS the architecture.               ║
║                                                            ║
║  HUMAN DECISION NODES:                                     ║
║  1. Fixed vs. adaptive circuit breaker thresholds          ║
║     → AI proposed fixed 2.0×. Rejected: task variance      ║
║       profiles differ. Adaptive thresholds are correct.    ║
║                                                            ║
║  2. LLM self-classification vs. HITL calibration           ║
║     → AI proposed runtime LLM classification. Rejected:    ║
║       adding cost to measure cost violates the chapter's   ║
║       own principle. Design-time tags + HITL calibration   ║
║       for ambiguous cases.                                 ║
║                                                            ║
║  FAILURE MODE:                                             ║
║  Wrong metric (CPI) → no visibility into decision-level    ║
║  cost → no model routing → uniform frontier model usage    ║
║  → budget blown at scale → project canceled by finance.    ║
║                                                            ║
║  EXERCISE:                                                 ║
║  Remove tiered routing. CPI stays flat. DpD explodes.      ║
║  Circuit breaker never trips. The blind spot is real.      ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║  CHAPTER 14: WHAT GETS MEASURED GETS MANAGED               ║
║  The Architectural Argument                                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  MASTER CLAIM:                                             ║
║  Architecture is the leverage point, not the model.        ║
║                                                            ║
║  THIS CHAPTER'S INSTANCE:                                  ║
║  The unit of cost measurement determines whether an        ║
║  agentic system is economically viable at scale.           ║
║  Cost-per-inference hides compounding. Dollar-per-decision ║
║  reveals it. The metric IS the architecture.               ║
║                                                            ║
║  HUMAN DECISION NODES:                                     ║
║  1. Fixed vs. adaptive circuit breaker threshol